# Chapter 6 — Top-down Ontology Development
### Notebook 4 · Agentic lab — choosing relations, and asking well

*Book reference: Extends §6.1–6.2*

Two things worth doing here. An agent that stops collapsing seven relations into one — and an MDP that **derives DOLCE's decision tree** from a reward function instead of taking it on authority.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch06_toolkit as ch6
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import ch06_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


**By the end of this notebook you can:**

1. Build an agent that distinguishes parthood from its impostors.
2. Pair every class in a dataset across the split, and see why an unpaired class is unlearnable.
3. Model **diagnosis** as an MDP with stochastic answers, and read the optimal policy as a decision tree.
4. Show that the derived tree matches the one a foundational ontology ships — and explain what changes it.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

`check_chaining` is the one that earns its place: an agent that calls it cannot produce the hand-in-the-orchestra inference, whatever it believes.

In [4]:
ctx = AG.Ch6Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:26s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":26s} {t.description.splitlines()[0]}')

foundational_categories    []
                           List the foundational categories with glosses and examples.
list_decision_questions    []
                           The decision questions used to place a class in a foundational category.
ask_decision_question      ['question', 'answer']
                           Record an answer to a foundational-category decision question.
part_whole_relations       []
                           List the part-whole relations, with which are parthood and which transitive.
classify_relation          ['part_category', 'whole_category', 'separable']
                           Pick the part-whole relation from the categories of part and whole.
check_chaining             ['first', 'second']
                           Decide whether two part-whole statements may be composed.


In [5]:
print(tools['classify_relation'].invoke(
    {'part_category': 'amount-of-matter', 'whole_category': 'physical-object'}))
print(tools['check_chaining'].invoke(
    {'first': 'component-of', 'second': 'member-of'}))
print()
for q, a in [('happens', 'false'), ('spatial', 'true'), ('mass', 'true')]:
    print(tools['ask_decision_question'].invoke({'question': q, 'answer': a}))

{"relation": "constituted-of", "parthood": false, "transitive": false, "example": "a statue is constituted of clay"}
{"valid": false, "reason": "'component of' and 'member of' are different relations; transitivity is a property of one relation, not of 'part of' in general"}

{"answers": {"happens": false}, "candidates": ["physical-object", "amount-of-matter", "feature", "quality", "abstract"]}
{"answers": {"happens": false, "spatial": true}, "candidates": ["physical-object", "amount-of-matter", "feature"]}
{"answers": {"happens": false, "spatial": true, "mass": true}, "candidates": ["amount-of-matter"]}


## 2. The dataset

Sixteen statements, all phrased with the words "part of". The cases are listed as **adjacent pairs of the same relation** because the split alternates — so every relation appears in both halves.

That pairing is not cosmetic. Chapter 5's Exercise 4.1 showed a rule becoming unlearnable when the training data could not exercise it; here the same risk applies to every one of the seven relations at once.

In [6]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print(pd.DataFrame([{'id': e.id, 'relation': e.gold_relation,
                     'parthood': e.gold_parthood,
                     'split': 'train' if e in train else 'dev'}
                    for e in AG.build_dataset('all')]).to_string(index=False))

                id        relation  parthood split
         wheel-car    component-of      True train
        door-house    component-of      True   dev
       tree-forest       member-of      True train
musician-orchestra       member-of      True   dev
      alcohol-wine sub-quantity-of      True train
    water-solution sub-quantity-of      True   dev
         gold-ring  constituted-of     False train
       clay-statue  constituted-of     False   dev
         lion-hunt participates-in     False train
 surgeon-operation participates-in     False   dev
        coffee-cup    contained-in     False train
   letter-envelope    contained-in     False   dev
    chewing-eating     involved-in      True train
 breathing-singing     involved-in      True   dev
   giraffe-reserve      located-in     False train
    village-valley      located-in     False   dev


In [7]:
print('train relations:', sorted({e.gold_relation for e in train}))
print('dev   relations:', sorted({e.gold_relation for e in dev}))
assert {e.gold_relation for e in train} == {e.gold_relation for e in dev}
print('\nEvery relation appears on both sides. Without that, the rules for the\n'
      'dev-only relations could never be learned and the held-out score would\n'
      'be capped for a reason invisible in the report.')

train relations: ['component-of', 'constituted-of', 'contained-in', 'involved-in', 'located-in', 'member-of', 'participates-in', 'sub-quantity-of']
dev   relations: ['component-of', 'constituted-of', 'contained-in', 'involved-in', 'located-in', 'member-of', 'participates-in', 'sub-quantity-of']

Every relation appears on both sides. Without that, the rules for the
dev-only relations could never be learned and the held-out score would
be capped for a reason invisible in the report.


## 3. Baseline: the single-`partOf` modeller

The un-instructed agent does what a great many published ontologies do — answers `component-of` for everything and calls it parthood.

In [8]:
lm = llm.configure_dspy(AG.PARTWHOLE_RULEBOOK, AG.partwhole_responder)
baseline = AG.PartWholeProgram()
for e in dev[:4]:
    p = baseline(**e.inputs())
    print(f'{e.id:20s} {e.statement}')
    print(f'{"":20s} answered {p.relation} / parthood={p.is_parthood}'
          f'   (gold {e.gold_relation} / {e.gold_parthood})')

door-house           A door is part of a house.
                     answered component-of / parthood=true   (gold component-of / True)
musician-orchestra   A musician is part of an orchestra.
                     answered component-of / parthood=true   (gold member-of / True)
water-solution       The water is part of the solution.
                     answered component-of / parthood=true   (gold sub-quantity-of / True)
clay-statue          The clay is part of the statue.
                     answered component-of / parthood=true   (gold constituted-of / False)


In [9]:
before = ev.evaluate_dataset(baseline, dev, AG.partwhole_scorer)
print('BEFORE:', before['mean_score'])
print('violations:', before['violations'])

BEFORE: 0.3125
violations: {'check-genuine-parthood': 4, 'member-for-collections': 1, 'subquantity-for-amounts': 1, 'constitution-not-parthood': 1, 'participation-not-parthood': 1, 'containment-not-parthood': 1, 'involvement-for-processes': 1, 'location-not-parthood': 1}


In [10]:
gepa_metric = ev.make_gepa_metric(AG.partwhole_scorer, AG.PARTWHOLE_RULEBOOK)
reflect = llm.reflection_lm(AG.PARTWHOLE_RULEBOOK, AG.partwhole_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=100, reflection_lm=reflect)
result = opt.compare(AG.PartWholeProgram(), tuned, dev, AG.partwhole_scorer)
print(result.report())

2026/08/17 07:51:50 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 100 metric calls of the program. This amounts to 6.25 full evals on the train+val set.


2026/08/17 07:51:50 INFO dspy.teleprompt.gepa.gepa: Using 8 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/100 [00:00<?, ?rollouts/s]

2026/08/17 07:51:50 INFO dspy.evaluate.evaluate: Average Metric: 2.5 / 8 (31.2%)


2026/08/17 07:51:50 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.3125


...

2026/08/17 07:51:51 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/17 07:51:51 INFO dspy.teleprompt.gepa.gepa: Iteration 26: All subsample scores perfect. Skipping.


2026/08/17 07:51:51 INFO dspy.teleprompt.gepa.gepa: Iteration 26: Reflective mutation did not propose a new candidate


GEPA Optimization:  98%|█████████▊| 98/100 [00:01<00:00, 76.40rollouts/s]

mean score  0.312  ->  1.000   (delta +0.688)
violations  {'check-genuine-parthood': 4, 'member-for-collections': 1, 'subquantity-for-amounts': 1, 'constitution-not-parthood': 1, 'participation-not-parthood': 1, 'containment-not-parthood': 1, 'involvement-for-processes': 1, 'location-not-parthood': 1}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,9 @@
 You are an ontology engineer. Say which part-whole relation the statement expresses.
+- RULE location-not-parthood: When the whole is a spatial region, the relation is located-in and it is NOT parthood.
+- RULE check-genuine-parthood: Decide separately whether the relation is genuine mereological parthood; containment, constitution, location and participation are not.
+- RULE containment-not-parthood: When the part could be removed leaving the whole unchanged, the relation is contained-in and it is NOT parthood.
+- RULE member-for-collections: When the whole is a collection (an orchestra, a f

In [11]:
found = AG.PARTWHOLE_RULEBOOK.active_in(result.instruction_after)
print(f'rules discovered: {len(found)}/{len(AG.PARTWHOLE_RULEBOOK.ids)}')
for rule_id in sorted(found):
    print('  -', rule_id)
print('missed:', sorted(set(AG.PARTWHOLE_RULEBOOK.ids) - found) or 'none')

rules discovered: 8/8
  - check-genuine-parthood
  - constitution-not-parthood
  - containment-not-parthood
  - involvement-for-processes
  - location-not-parthood
  - member-for-collections
  - participation-not-parthood
  - subquantity-for-amounts
missed: none


> The rule worth noticing is `check-genuine-parthood`. The others fix *which relation* is named; that one fixes whether **anything may be inferred from it**. An agent that gets the name right and parthood wrong will still license the bad chain.

## 4. Deriving the decision tree

Now the interesting MDP. To align a class the agent asks yes/no questions, each costing a little, and commits when more questioning is not worth the price.

| | |
|---|---|
| **S** | the categories still consistent with the answers so far |
| **A** | ask a question that actually splits the set, or commit |
| **T** | **stochastic** — you do not know the answer until you ask |
| **R** | −cost per question; on commit, the probability of being right (`1/k`) |

Committing with `k` candidates left is right with probability `1/k`, so the agent genuinely trades questions against accuracy.

In [12]:
M = AG.CategoryDiagnosisMDP(question_cost=0.05)
print('reachable candidate sets:', len({s.candidates for s in M.states()}))
print('states (with commit flag):', len(M.states()))
print('\nNote: the full power set of 7 categories would be 128 subsets.\n'
      'Only the ones actually reachable by asking questions are enumerated.')

reachable candidate sets: 33
states (with commit flag): 66

Note: the full power set of 7 categories would be 128 subsets.
Only the ones actually reachable by asking questions are enumerated.


In [13]:
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'V*(s0) = {V[s0]:.4f}')
print(f'  (1.0 accuracy minus the expected cost of the questions asked)')

V*(s0) = 0.8500
  (1.0 accuracy minus the expected cost of the questions asked)


### The optimal policy, rendered as the tree it is:

In [14]:
for line in M.decision_tree(pi):
    print(line)

happens? (Does it happen or unfold in time, rather than merely exist through time?)
  yes:
    telic? (Does it have a natural endpoint or culmination?)
      yes:
        -> ['event']
      no:
        -> ['process']
  no:
    spatial? (Does it have a location in space?)
      yes:
        mass? (Is any part of it describable by the same term (is it a mass noun)?)
          yes:
            -> ['amount-of-matter']
          no:
            dependent? (Must it inhere in, or belong to, some other entity to exist?)
              yes:
                -> ['feature']
              no:
                -> ['physical-object']
      no:
        dependent? (Must it inhere in, or belong to, some other entity to exist?)
          yes:
            -> ['quality']
          no:
            -> ['abstract']


> **Compare that with §6.1.** The optimiser split on `happens?` first — the endurant/perdurant distinction, which is exactly where every foundational ontology starts. Nobody told it that; it followed from the question being the one that best halves the candidate set.

This is the most satisfying result in the course: a decision tree that textbooks present as received wisdom, **derived** from a cost model.

In [15]:
import random
random.seed(0)
lengths = []
for _ in range(400):
    ep = mdp.run_episode(M, mdp.greedy_policy(pi))
    lengths.append(len(ep) - 1)      # questions asked before committing
print(f'average questions asked: {sum(lengths)/len(lengths):.2f}')
print(f'range: {min(lengths)}-{max(lengths)}')
print('\nA perdurant is settled in two questions; an endurant needs three or\n'
      'four. The tree is unbalanced because the categories are.')

average questions asked: 3.02
range: 2-4

A perdurant is settled in two questions; an endurant needs three or
four. The tree is unbalanced because the categories are.


### Exercise 4.1 — Make questions expensive

Raise `question_cost` until the optimal policy stops asking altogether. Report the threshold and explain it in terms of the accuracy being bought.

In [16]:
# YOUR CODE HERE


<details>
<summary>Solution 4.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [17]:
rows = []
for cost in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    Mc = AG.CategoryDiagnosisMDP(question_cost=cost)
    Vc, pic = mdp.value_iteration(Mc)
    tree = Mc.decision_tree(pic)
    asks = sum(1 for line in tree if '?' in line)
    rows.append({'question_cost': cost,
                 'V*': round(Vc[Mc.initial_state()], 4),
                 'questions in tree': asks})
print(pd.DataFrame(rows).to_string(index=False))
silent = [r for r in rows if r['questions in tree'] == 0]
print(f"\nthe agent stops asking at cost >= {silent[0]['question_cost'] if silent else 'never in this range'}")
print('Committing blind to one of seven categories is worth 1/7 = 0.143. A\n'
      'question is worth asking only while it buys more accuracy than it costs,\n'
      'so once questions get expensive enough the optimal ontologist guesses --\n'
      'which is a statement about budgets, not about rigour.')

 question_cost     V*  questions in tree
          0.00 1.0000                  6
          0.05 0.8500                  6
          0.10 0.7000                  6
          0.20 0.4000                  6
          0.30 0.1429                  0
          0.50 0.1429                  0

the agent stops asking at cost >= 0.3
Committing blind to one of seven categories is worth 1/7 = 0.143. A
question is worth asking only while it buys more accuracy than it costs,
so once questions get expensive enough the optimal ontologist guesses --
which is a statement about budgets, not about rigour.


### Exercise 4.2 — Remove a question and watch the tree adapt

Drop `happens` from the question set and re-derive the tree. Report what it costs in expected value, and what the new first question is.

In [18]:
# YOUR CODE HERE


<details>
<summary>Solution 4.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [19]:
reduced = tuple(q for q in ch6.DECISION_QUESTIONS if q != 'happens')
M2 = AG.CategoryDiagnosisMDP(question_cost=0.05, questions=reduced)
V2, pi2 = mdp.value_iteration(M2)
print('remaining questions:', reduced)
print(f'V* with all five  : {V[s0]:.4f}')
print(f'V* without happens: {V2[M2.initial_state()]:.4f}')
print('\nnew tree:')
for line in M2.decision_tree(pi2):
    print(line)
assert V2[M2.initial_state()] <= V[s0]
print('\nThe endurant/perdurant question is the most informative single question\n'
      'available, so removing it costs value -- but the optimiser simply\n'
      're-plans around the loss rather than failing. That is the practical\n'
      'argument for deriving a decision procedure instead of hard-coding one:\n'
      'when the available evidence changes, the procedure should change with it.')

remaining questions: ('spatial', 'mass', 'dependent', 'telic')
V* with all five  : 0.8500
V* without happens: 0.7071

new tree:
spatial? (Does it have a location in space?)
  yes:
    mass? (Is any part of it describable by the same term (is it a mass noun)?)
      yes:
        -> ['amount-of-matter']
      no:
        dependent? (Must it inhere in, or belong to, some other entity to exist?)
          yes:
            -> ['feature']
          no:
            telic? (Does it have a natural endpoint or culmination?)
              yes:
                -> ['event']
              no:
                -> ['physical-object', 'process']
  no:
    dependent? (Must it inhere in, or belong to, some other entity to exist?)
      yes:
        -> ['quality']
      no:
        -> ['abstract']

The endurant/perdurant question is the most informative single question
available, so removing it costs value -- but the optimiser simply
re-plans around the loss rather than failing. That is the practical
argum

### Exercise 4.3 — Give the agent the chaining tool and prove it cannot be fooled

Show that an agent using `check_chaining` refuses the hand/orchestra inference, and that the refusal is grounded in the relation properties rather than in the prompt.

In [20]:
# YOUR CODE HERE


<details>
<summary>Solution 4.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [21]:
ctx2 = AG.Ch6Context()
t2 = {t.name: t for t in AG.build_toolset(ctx2)}

hand = json.loads(t2['classify_relation'].invoke(
    {'part_category': 'physical-object', 'whole_category': 'physical-object'}))
musician = json.loads(t2['classify_relation'].invoke(
    {'part_category': 'physical-object', 'whole_category': 'collection'}))
print('hand -> musician  :', hand['relation'], '(parthood', hand['parthood'], ')')
print('musician -> orch. :', musician['relation'], '(parthood', musician['parthood'], ')')

chain = json.loads(t2['check_chaining'].invoke(
    {'first': hand['relation'], 'second': musician['relation']}))
print('\nchain valid?', chain['valid'])
print('reason      :', chain['reason'])
assert not chain['valid']
print('\ntool calls:', ctx2.log.names())
print('\nThe refusal comes from the relation table, not from an instruction\n'
      'telling the agent that orchestras are special. Encoding a distinction in\n'
      'a TOOL rather than a PROMPT is what makes it survive prompt optimisation,\n'
      'model swaps, and the next engineer.')

hand -> musician  :

 component-of (parthood True )
musician -> orch. : member-of (parthood True )

chain valid? False
reason      : 'component of' and 'member of' are different relations; transitivity is a property of one relation, not of 'part of' in general

tool calls: ['classify_relation', 'classify_relation', 'check_chaining']

The refusal comes from the relation table, not from an instruction
telling the agent that orchestras are special. Encoding a distinction in
a TOOL rather than a PROMPT is what makes it survive prompt optimisation,
model swaps, and the next engineer.


## Chapter 6 in the course arc

| | Ch. 3 | Ch. 4 | Ch. 5 | Ch. 6 |
|---|---|---|---|---|
| MDP | budgeted, stochastic | construction | plan under prerequisites | **diagnosis (derives a decision tree)** |
| grader | free oracle | profile table | measured CQ coverage | relation taxonomy |
| the error it prevents | unsound entailment | profile violation | ontologically wrong axiom | **unsound part-whole chaining** |

Chapters 5 and 6 together make one argument. Chapter 5 found an error a reasoner could not see; Chapter 6 supplied the vocabulary to repair it. Neither was a matter of more logic — both were a matter of **making more distinctions**.

That is the top-down half of ontology development. Chapter 7 goes the other way: extracting an ontology from text and data, where you get no distinctions for free at all.